[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import math

In [24]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    b, o, v = Q.shape
    logits = torch.einsum('bov,biv->boi', Q, K) / math.sqrt(v)
    # logits = torch.tril(logits)
    # logits[logits==0] = float('-inf')
    mask = torch.triu(torch.ones(o, o, dtype=torch.bool), diagonal=1)   # True where j > i
    logits.masked_fill_(mask,float('-inf'))
    attn = torch.softmax(logits,dim=-1)
    output = torch.einsum('boi,biv->bov', attn, V)
    return output
    # pass  # Replace this

In [25]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [26]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.7ms)
  ✅ [2/4] Future tokens don't affect past (6.3ms)
  ✅ [3/4] First position only sees itself (3.2ms)
  ✅ [4/4] Gradient flow (1.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (15.1ms total)
  Progress saved. Run status() to see your dashboard.

